# v3 on held-out in-distribution logs

**The question:** how good is v3 on exactly the formats it was trained for,
when the specific lines are new?

500 lines, 50 from each of the ten Loghub systems v3 trained on. The pool is
Loghub-**1.0**; v3 trained on Loghub-**2.0**. The builder verifies the
intersection is empty rather than assuming it — it reports
`excluded as already-seen: 0`, and `measure_contamination.py` independently
confirms **0/500** template overlap.

`rule_parser.py` returns a parse on **500/500** here, against 1/20 on
`messy.log`. That contrast is the definition of in-distribution.

**Runtime → Change runtime type → T4 GPU.** Four cells, ~5 min, $0.

In [ ]:
# 1. setup (~3 min) -- restarts once, then re-run this same cell
import os
os.chdir('/content')
!rm -rf tiny-log-parser
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
os.chdir('/content/tiny-log-parser')
!pip install -q "transformers==4.51.3" "peft==0.20.0" accelerate bitsandbytes openai
!pip uninstall -q -y torchao      # peft 0.20 raises on torchao < 0.16; Colab ships 0.10

import importlib.util, transformers, peft
stale = ((transformers.__version__, peft.__version__) != ("4.51.3", "0.20.0")
         or importlib.util.find_spec("torchao") is not None)
if stale:
    print("restarting to pick up the new versions...")
    os.kill(os.getpid(), 9)

In [ ]:
# 2. prove the split is clean BEFORE looking at any accuracy number
import os
os.chdir('/content/tiny-log-parser')
!python3 v3/measure_contamination.py --eval real-eval/corpus_heldout.jsonl

Expect `0/500` in both the `exact` and `same tmpl` columns. That is the
reply to "you benchmarked on your training data" — made before the result
is known, not after.

In [ ]:
# 3. v3 on the 500 (~2 min, mostly model load)
!python3 real-eval/predict.py --arm model --adapter arshirazi/tiny-log-parser-v3 \
    --corpus real-eval/corpus_heldout.jsonl --batch 32 \
    --out real-eval/preds_heldout_v3.jsonl

Watch the `unparseable` count. On in-distribution formats it should be 0 —
v3 emitted 1 unparseable in 262 lines on P1.

In [ ]:
# 4. v3 against the rules reference -- only the lines they disagree on
!python3 real-eval/compare_arms.py --corpus real-eval/corpus_heldout.jsonl \
    --only-disagreements \
    rules=real-eval/preds_heldout_rules.jsonl v3=real-eval/preds_heldout_v3.jsonl

---

## Bring your own logs

Paste lines from your own Hadoop, Spark, Linux, HDFS, Apache, Zookeeper,
HealthApp, OpenSSH, OpenStack or Proxifier deployment. These are the ten
systems v3 was trained on, so this is the case it should handle well.

No labels are needed and none are invented — you read the output and judge it
against `schema_v2.SPEC_EVAL` yourself.

In [ ]:
# 5. your own logs -- one line per line, no labels needed
open('mine.log', 'w').write('''\
2015-10-18 18:08:35,537 WARN [LeaseRenewer:msrabi@msra-sa-41:9000] org.apache.hadoop.hdfs.LeaseRenewer: Failed to renew lease
17/06/09 20:10:58 INFO mapred.SparkHadoopMapRedUtil: attempt_201706092018_0024_m_000161_1156: Committed
Nov 22 14:31:57 combo kernel: get_user_size+0x30/0x57
081111 041509 20066 INFO dfs.DataNode$PacketResponder: Received block blk_-8965508153794244104
''')

!python3 real-eval/messy_corpus.py mine.log -o real-eval/corpus_mine.jsonl
!python3 real-eval/predict.py --arm rules --corpus real-eval/corpus_mine.jsonl \
    --out real-eval/preds_mine_rules.jsonl
!python3 real-eval/predict.py --arm model --adapter arshirazi/tiny-log-parser-v3 \
    --corpus real-eval/corpus_mine.jsonl --batch 32 \
    --out real-eval/preds_mine_v3.jsonl

In [ ]:
# 6. v3 and the parser, side by side, on your lines
!python3 real-eval/compare_arms.py --corpus real-eval/corpus_mine.jsonl \
    rules=real-eval/preds_mine_rules.jsonl v3=real-eval/preds_mine_v3.jsonl

### What to expect, honestly

- **This is inspection, not measurement.** No gold labels, so nothing here is
  an accuracy number. Agreement between the two arms is not proof either —
  both can be wrong on the same line.
- **Your format may not be Loghub's format.** v3 learned *Loghub's* Hadoop, not
  every Hadoop. A different appender layout is a different distribution, and
  the ten system names are not a guarantee.
- **Off these ten formats it degrades, and we know where.** JSON, logfmt and
  Apache combined access logs appear **zero** times in v3's 10,728 training
  examples. So do `<PRI>` syslog prefixes, epoch timestamps and `tid=<32 hex>`.
  Feed it those and it will show.
- **`latency_ms`, `trace_id` and `status_code` are the fragile three.** Non-null
  in 2.97%, 3.17% and 1.79% of training, so v3's learned prior is to abstain.
  Right on this distribution, wrong off it. Every non-null latency it ever saw
  used the notation `time: <n>` — `took=`, `rt=`, `elapsed=` and `duration_us`
  are untrained territory.

The parser (`rules`) is the useful contrast: it scores 100% on these ten
formats and fails to parse 19 of 20 lines in `messy.log`. It is a hand-written
ceiling that does not move. That is the thing the fine-tune is trying to buy
its way out of.

## Reading the result

**There are no gold labels here and none are invented.** The method is
adjudicated disagreement: diff the two arms, hand-check only where they differ.

That is licensed by one fact and does not generalise: `rule_parser.py` scored
**100.0% (96/96)** on the P1 in-distribution slice against independent hand
labels. It is a reference *on this distribution* and nowhere else — on
`messy.log` it fails to parse 19 of 20 lines.

- **Agreement is a proxy, not a measurement.** Both arms can be wrong on the
  same line.
- **Every disagreement needs a human.** Rules being 96/96 on one sample does
  not make it right on line 497. Judge against `schema_v2.SPEC_EVAL` and
  `ADJUDICATION.md`; expect a handful, and expect some to be rules' fault.
- **Watch `status_code`, `trace_id`, `latency_ms`.** Non-null in 1.79%, 3.17%
  and 2.97% of training. v3's learned prior is to abstain — right on this
  distribution, wrong off it. A disagreement here is the interesting one.

Calibration: v3 scored **99.0% exact (95/96)** on the P1 in-distribution
slice, Gemini 90.6%. If this run lands far from ~97-99%, suspect the run.

## Next: the frontier arms

To turn this into a head-to-head, run the same corpus through the three
frontier models (needs an OpenRouter key, ~$2 total):

```python
import getpass, os
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()

for model, tag in [('google/gemini-3.1-pro-preview', 'gemini'),
                   ('anthropic/claude-opus-4.8',     'claude'),
                   ('openai/gpt-5.2',                'gpt')]:
    !python3 real-eval/predict.py --arm gemini --gemini-model {model} --shots 0 \
        --corpus real-eval/corpus_heldout.jsonl \
        --out real-eval/preds_heldout_{tag}.jsonl
```

Then adjudicate only the lines where the arms disagree, and score.